# Flight Trend & Optimization Analyzer

## 1. Project Overview

This project fetches **real-time global flight data** from the [OpenSky Network API](https://opensky-network.org) and performs data analysis using Object-Oriented Programming (OOP) in Python.

**Core objectives:**
1. Fetch live flight data via REST API
2. Analyze country distribution and airline activity
3. Study velocity and altitude patterns
4. Filter and export data for further use


---
## 2. Environment Setup & Module Import


In [ ]:
import pandas as pd
from src.flight_logic import FlightDataAnalyzer, LiveFlightAPI

print("[INFO] Environment initialized. Modules loaded: FlightDataAnalyzer, LiveFlightAPI.")


---
## 3. Load Live Data from OpenSky Network API

We use the `LiveFlightAPI` child class (inherits from `FlightDataAnalyzer`) to fetch real-time flight data.
The endpoint `https://opensky-network.org/api/states/all` delivers a dynamic **JSON response**:
- The root object contains `time` (Unix timestamp) and `states` (a 2D array of airborne aircraft state vectors).
- Because `states` contains an array of raw values rather than named keys, our ingestion logic parses the official index positions:
  - `flight[0]`: **icao24** (unique 24-bit transponder hex ID)
  - `flight[1]`: **callsign** (8-character flight identifier)
  - `flight[2]`: **country** (aircraft country of registration)
  - `flight[5]`: **longitude** & `flight[6]`: **latitude** (WGS-84 coordinates)
  - `flight[7]`: **altitude** (barometric altitude in meters)
  - `flight[9]`: **velocity** (ground speed in m/s)
  - `flight[10]`: **heading** (direction of motion in degrees)

The parsed records are converted into a `pandas.DataFrame` and cleaned of missing transponder records.


In [ ]:
# Initialize API loader and fetch live flight data
live_api = LiveFlightAPI()

if live_api.fetch_live_flights(limit=500):
    print(f"[SUCCESS] Loaded {len(live_api.df)} live flights from OpenSky Network")
    print(f"Columns: {list(live_api.df.columns)}")
    display(live_api.df.head(10))
else:
    print("[ERROR] Failed to fetch data. Check network connection.")


---
## 4. Feature 1: Country Distribution Analysis

Which countries have the most airborne flights right now?


In [ ]:
# Top 10 countries by live flight count (tabular analysis)
country_stats = live_api.df["country"].value_counts().head(10)

print("=== Top 10 Countries by Live Flight Count ===")
for country, count in country_stats.items():
    print(f"  {country}: {count} flights")


---
## 5. Feature 2: Velocity & Altitude Analysis

Analyze flight speed and altitude patterns across countries.


In [ ]:
# Compute velocity and altitude stats by country
valid_flights = live_api.df.dropna(subset=["velocity", "altitude"])

vel_stats = valid_flights.groupby("country").agg(
    flight_count=("icao24", "count"),
    avg_velocity=("velocity", "mean"),
    avg_altitude=("altitude", "mean"),
    max_velocity=("velocity", "max")
).reset_index()

vel_stats["avg_velocity"] = vel_stats["avg_velocity"].round(2)
vel_stats["avg_altitude"] = vel_stats["avg_altitude"].round(2)
vel_stats["max_velocity"] = vel_stats["max_velocity"].round(2)

top_vel = vel_stats.sort_values("flight_count", ascending=False).head(10)
print("=== Top 10 Countries: Velocity & Altitude Stats ===")
display(top_vel)


---
## 6. For-loop: Detailed Flight Summary by Country

Use explicit `for` loops to iterate and print detailed stats for each top country.


In [ ]:
# For-loop: print detailed summary for top countries
top_n = 5
country_counts = live_api.df["country"].value_counts().head(top_n)

print(f"\n{'='*60}")
print(f"  Top {top_n} Countries - Detailed Flight Summary")
print(f"{'='*60}")

for country, count in country_counts.items():
    subset = live_api.df[live_api.df["country"] == country]
    avg_vel = subset["velocity"].mean() if not subset["velocity"].isna().all() else 0
    avg_alt = subset["altitude"].mean() if not subset["altitude"].isna().all() else 0
    print(f"\nCountry: {country}")
    print(f"  Total flights: {count}")
    print(f"  Avg velocity: {avg_vel:.2f} m/s")
    print(f"  Avg altitude: {avg_alt:.2f} m")

print(f"\n{'='*60}")


---
## 7. For-loop: Count Flights by Airline (Callsign)

Count how many flights each airline (callsign) currently has in the air.


In [ ]:
# For-loop: count flights by callsign
airline_counts = {}
grouped = live_api.df.groupby("callsign")

for airline_name, group in grouped:
    if airline_name and airline_name.strip() and airline_name != "N/A":
        airline_counts[airline_name] = len(group)

sorted_airlines = dict(sorted(airline_counts.items(), key=lambda x: x[1], reverse=True))

print(f"\n{'='*60}")
print("  Top Airlines by Active Flight Count")
print(f"{'='*60}")

count = 0
for airline, flights in sorted_airlines.items():
    if count >= 10:
        break
    print(f"  {airline}: {flights} flights")
    count += 1

print(f"{'='*60}\n")


---
## 8. Filter by Country

Filter live flights for a specific country and analyze distribution.


In [ ]:
# Filter flights for United States
us_flights = live_api.get_flights_by_country("United States")

if not us_flights.empty:
    print(f"=== US Flights ({len(us_flights)} total) ===")
    display(us_flights.head(10))
else:
    print("[WARNING] No US flights found in current data")


In [ ]:
live_api.show_callsign_stats()
